# Article collection | Web corpus

This notebook documents a **two-stage Selenium collection process**: discover linked articles and retrieve their text. It records source section, topic, title, URL, and body. Use the bundled CSV for reproducible experiments; a new crawl may yield different material because the website can change.

**Reading note:** Check permission, robots instructions, request volume, and any personal information before collecting or redistributing text. The CSV contains the originally supplied material with new column headings only.


## Collect article metadata and text

Use a headless browser to discover category listings, collect article links, and extract page text. The scraper writes the same five kinds of fields to the revised CSV name. Website markup can change, so this step is optional if you use the bundled snapshot.


In [ ]:
# Configure the headless browser, traverse available listing pages, and extract article metadata.
# Write the article corpus with revised header labels while preserving the existing extraction strategy.

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd


options = Options()

options.add_argument("--headless")

options.add_argument("--no-sandbox")

options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)


titles, categories, topics, texts, urls_to_visit = [], [], [], [], []

seen_urls = set()


print("Step 1: Finding all listing pages...")

driver.get("https://spunout.ie/information")

time.sleep(3)

all_links = driver.find_elements(By.XPATH, "//a[contains(@href, '/category/')]")

raw_listing_urls = list(set([el.get_attribute("href") for el in all_links]))


for listing_url in raw_listing_urls:
    print(f"\n[SCANNING LISTING] {listing_url}")
    
    driver.get(listing_url)
    
    
    while True:
        
        time.sleep(2)
        
        blocks = driver.find_elements(By.CLASS_NAME, "news_list_single")
        
        if not blocks: break
        
        
        for block in blocks:
            try:
                
                article_url = block.find_element(By.TAG_NAME, "a").get_attribute("href")
                
                
                if article_url not in seen_urls:
                    
                    title = block.find_element(By.TAG_NAME, "h5").text
                    
                    
                    
                    path_parts = article_url.replace("https://spunout.ie/", "").strip('/').split('/')
                    
                    
                    
                    final_cat = path_parts[0].replace('-', ' ').title().replace(' ', '_')
                    
                    
                    final_topic = ""
                    if len(path_parts) >= 3:
                        final_topic = path_parts[1].replace('-', ' ').title().replace(' ', '_')
                    
                    
                    print(f"  [ADDED] {article_url}")
                    print(f"   -> Category: {final_cat} | Topic: {final_topic if final_topic else '[Empty]'}")
                    
                    
                    titles.append(title)
                    categories.append(final_cat)
                    topics.append(final_topic)
                    urls_to_visit.append(article_url)
                    
                    seen_urls.add(article_url)
            except:
                
                continue

        
        try:
            
            next_btn = driver.find_element(By.CSS_SELECTOR, "a.next")
            
            
            driver.execute_script("arguments[0].click();", next_btn)
        except:
            
            break


print(f"\nStep 3: Extracting text from {len(urls_to_visit)} articles...")

for i, url in enumerate(urls_to_visit):
    try:
        
        driver.get(url)
        
        
        content = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "content_inner"))
        ).text
        
        
        texts.append(" ".join(content.split()))
        
        if (i+1) % 10 == 0: print(f"Progress: {i+1}/{len(urls_to_visit)}")
    except:
        
        texts.append("N/A")


driver.quit()



df = pd.DataFrame({
    "article_title": titles,
    "source_section": categories,
    "source_topic": topics,
    "article_body": texts,
    "source_url": urls_to_visit
})


df.to_csv("./data/youth_information_articles.csv", index=False, encoding='utf-8-sig')
print("\nSuccess! CSV saved.")

**How to interpret this step.** Inspect the collected URLs and article-body completeness before analyzing any text. The accompanying CSV is a fixed historical snapshot; scraping again can produce different rows.
